In [1]:
import numpy as np
import sys
sys.path.append('../')
import pandas as pd
import scanpy as sc

adata = sc.read_h5ad(
    "../data/obesity_challenge_2.h5ad"
)

test_group = pd.read_csv(
    "../data/predict_perturbations_2.txt",
    header=None
)[0].tolist()

In [2]:
import sys
sys.path.append('../')
from scripts.pairing import  load_gene_embeddings

from scripts.pairing import get_condition_encoder

emb = load_gene_embeddings("large",resource_dir="../resources")

encode_fn = get_condition_encoder(
    embedding_dict=emb,
    condition_mode="embedding",
    add_num_guides=False
)

pert_embeddings = {
    p: encode_fn(p)
    for p in adata.obs["gene"].unique()
}

test_embeddings = {
    p: encode_fn(p)
    for p in test_group
}

In [3]:
from sklearn.metrics.pairwise import cosine_distances
import pandas as pd
import numpy as np


all_perts = list(pert_embeddings.keys())

all_emb = np.vstack([
    pert_embeddings[p]
    for p in all_perts
])

test_emb = np.vstack([
    test_embeddings[p]
    for p in test_group
])


# distance matrix
dist_matrix = cosine_distances(
    test_emb,
    all_emb
)


val_matches = []

for i, test_pert in enumerate(test_group):

    distances = dist_matrix[i]

    df = pd.DataFrame({
        "pert": all_perts,
        "distance": distances
    })

    # 排除 test 自身
    df = df[~df["pert"].isin(test_group)]

    # 找最近的
    nearest = df.sort_values("distance").iloc[0]

    val_matches.append({
        "test_pert": test_pert,
        "val_pert": nearest["pert"],
        "distance": nearest["distance"]
    })


val_df = pd.DataFrame(val_matches)


# 去除重複（有些 test 可能 map 到同一個 val）
val_group = val_df["val_pert"].unique().tolist()


print("val size:", len(val_group))
print("\nvalidation perturbations:")
print(val_group)


print("\nmapping:")
print(val_df)

val size: 40

validation perturbations:
['CEBPD+CREB1', 'KIF11+MLXIPL', 'CEBPD+SREBF1', 'CEBPD+ZBED3', 'CEBPB+PPARG', 'CEBPB+FOXO1', 'CEBPA+MLXIPL', 'STAT5A+TCF7L2', 'MLXIPL+STAT5A', 'CEBPB+NC', 'CEBPD+PPARG', 'CEBPA+NC+NC', 'PPARG+ZBED3', 'PPARG+STAT5A', 'NR3C1+PPARG2', 'CREB1+STAT5A', 'KLF15+PPARG', 'PPARG2+SREBF1', 'CEBPA+STAT5B', 'CEBPB+KIF11', 'CEBPA+CREB1', 'MLXIPL+PPARG2', 'PPARG2+TCF7L2', 'KLF15+NC', 'CEBPA+SREBF1', 'CEBPB+CEBPB+KLF15', 'CEBPA+PPARG2', 'SF3B1+SF3B1', 'KIF11+KIF11', 'CEBPA+ZBED3', 'CEBPB+TCF7L2', 'NR3C1+PPARG', 'PPARG2+ZBED3', 'POLR2D+PPARG', 'CEBPB+STAT5B', 'STAT5A+ZBED3', 'CEBPB+NR3C1', 'STAT5B+STAT5B', 'SF3B1+SREBF1', 'STAT5A+STAT5B']

mapping:
        test_pert       val_pert  distance
0           CREB1    CEBPD+CREB1  0.098043
1          MLXIPL   KIF11+MLXIPL  0.089958
2          SREBF1   CEBPD+SREBF1  0.106645
3           CEBPD    CEBPD+ZBED3  0.104832
4          PPARG2    CEBPB+PPARG  0.051935
..            ...            ...       ...
57   SF3B1+TCF7L2  

In [4]:
pert_count = {
    pert: len(adata[adata.obs["gene"] == pert])
    for pert in all_perts
}
# minimum cell threshold
min_n = 50   # <- 自行設定


# 去除重複（有些 test 可能 map 到同一個 val）
val_group = val_df["val_pert"].unique().tolist()


# ===== 新增這段 =====
val_group = [
    p for p in val_group
    if pert_count.get(p, 0) >= min_n
]
# ===================


print("val size:", len(val_group))
print("\nvalidation perturbations:")
print(val_group)
val_df = val_df[val_df["val_pert"].isin(val_group)].reset_index(drop=True)


print("\nmapping:")
print(val_df)

val size: 34

validation perturbations:
['CEBPD+CREB1', 'KIF11+MLXIPL', 'CEBPD+SREBF1', 'CEBPD+ZBED3', 'CEBPB+PPARG', 'CEBPB+FOXO1', 'CEBPA+MLXIPL', 'STAT5A+TCF7L2', 'MLXIPL+STAT5A', 'CEBPB+NC', 'CEBPD+PPARG', 'PPARG+ZBED3', 'PPARG+STAT5A', 'NR3C1+PPARG2', 'CREB1+STAT5A', 'KLF15+PPARG', 'PPARG2+SREBF1', 'CEBPA+STAT5B', 'CEBPA+CREB1', 'MLXIPL+PPARG2', 'PPARG2+TCF7L2', 'KLF15+NC', 'CEBPA+SREBF1', 'CEBPA+PPARG2', 'CEBPA+ZBED3', 'CEBPB+TCF7L2', 'NR3C1+PPARG', 'PPARG2+ZBED3', 'POLR2D+PPARG', 'CEBPB+STAT5B', 'STAT5A+ZBED3', 'CEBPB+NR3C1', 'SF3B1+SREBF1', 'STAT5A+STAT5B']

mapping:
        test_pert       val_pert  distance
0           CREB1    CEBPD+CREB1  0.098043
1          MLXIPL   KIF11+MLXIPL  0.089958
2          SREBF1   CEBPD+SREBF1  0.106645
3           CEBPD    CEBPD+ZBED3  0.104832
4          PPARG2    CEBPB+PPARG  0.051935
..            ...            ...       ...
57   SF3B1+TCF7L2   CEBPB+TCF7L2  0.032195
58  MLXIPL+MLXIPL   KIF11+MLXIPL  0.064514
59  POLR2D+STAT5A  STAT5A+STAT5

In [7]:
val_df.to_csv(
    "../data/preprocessed/val_perts/validation_mapping.csv",
    index=False
)